# About the Notebook

This notebook provides a blueprint for building production-ready AI agents. It addresses the critical gap between _it worked on my machine_ and _it works reliably at scale_.

**The Problem:** Agents often fail silently in production. When LLM outputs vary slightly, JSON parsing breaks, or different providers return inconsistent types (strings instead of integers), it results in hours of debugging brittle code.

**The Solution:** By using **Pydantic models** as technical contracts, we define the exact shape of data at the system boundary. This prevents invalid state from propagating, simplifies provider transitions, and ensures the system is self-healing.

---

## Technical Pillars for Production

### 1. Deterministic Policy Management
To ensure a run is stable from start to finish, the validation policy must be **pinned at the start**.
* **Policy Hashing:** The active configuration is hashed and stored in the run state.
* **Consistency:** Even if global configurations change mid-deployment, ongoing runs follow the rules they were initialized with.
* **Audit Trails:** Every database record includes a `policy_hash`, allowing developers to see exactly which validation rules generated that specific data point.

### 2. Performance-Optimized Validation
Validation should not become a bottleneck.
* **Model Caching:** Pydantic models are built once per `(mode, policy_hash)` combination and cached, avoiding the overhead of class generation in "hot paths."
* **Direct Dictionary Passing:** Nodes pass raw data directly to factory functions, eliminating unnecessary parsing overhead within the execution graph.

### 3. Explicit State Management
Rather than inferring success from error counts, we use **boolean validation flags** (e.g., `prompt_ok`, `image_ok`).
* **Clear Failure Modes:** Routing logic checks flags directly.
* **Human-Readable Logs:** Retry logic increments first and then checks against limits, ensuring logs clearly state "Attempt 1" through "Attempt N."


### 4. Boundary Separation
We maintain a strict separation between **Tool Results** and **Database Records**.
* **Contract Independence:** The schema used to validate tool output can evolve independently from the schema used for permanent storage.
* **Type-Safe Contracts:** Database records accept validated `BaseModel` instances rather than raw dictionaries or `Any` types.

---

## Architecture Design Patterns

The implementation follows a **Three-Block Structure**:

| Component | Responsibility |
| :--- | :--- |
| **Governance Config** | Centralized policy management using OmegaConf for mode switching (Strict/Lenient). |
| **Contracts & Gates** | Cached Pydantic factories that act as the defensive boundary for untrusted data. |
| **Graph Wiring** | The LangGraph workflow that manages state persistence, checkpointers, and retry logic. |



---

## Pydantic vs. Instructor: Strategic Comparison

While Pydantic protects the system boundary within a graph, **Instructor** moves recovery closer to the source of generation.

| Feature | LangGraph + Pydantic | Instructor |
| :--- | :--- | :--- |
| **Recovery Logic** | Recovery lives in the **Graph** via refine nodes. | Recovery lives in the **Client** via automatic retries. |
| **Visibility** | Every retry step is a distinct node in the trace. | Retries are handled internally during the LLM call. |
| **Complexity** | Higher (requires conditional routing and loops). | Lower (linear flow from generation to next step). |
| **Best For** | Complex multi-agent systems and audit-heavy tasks. | Structured data extraction and simple "fix-it" loops. |


---

## Critical Implementation Details

This notebook demonstrates several production-critical fixes:
1. **Stable Schema Hashing:** Dynamically generated models use explicit titles to ensure JSON schema references remain stable across processes.
2. **Aggressive Normalization:** Validation keywords and policy values are sorted and lowercased before hashing to prevent unnecessary cache misses.
3. **Traceability:** A `run_id` field ties every database record back to a specific LangGraph `thread_id` and checkpoint state.


In [1]:
%%capture
!pip install langgraph langchain-core langchain-openai typing-extensions instructor


In [2]:
import os
from dotenv import load_dotenv


load_dotenv()

OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')

# Imports

In [15]:
from __future__ import annotations

# Standard library
import json
import hashlib
from datetime import datetime, timezone
from operator import add
from typing import Any, Annotated, Literal, Optional, List
from typing_extensions import TypedDict

# LangChain / LangGraph
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver, MemorySaver

# Third party
from omegaconf import OmegaConf, DictConfig
from pydantic import BaseModel, Field, ValidationError, field_validator
import instructor


## The Problem - Silent Failures

Let's start with a common scenario: an agent that processes user data. Without Pydantic, type mismatches fail silently or cause cryptic errors later.

Silent Failure - String vs Integer: Your database expects an integer, but the LLM returns a string.

In [4]:
def process_user_data_brittle(data: dict):
    """Without validation, this fails silently or crashes unpredictably."""
    user_id = data.get("user_id")  # Could be "123" (string) or 123 (int)
    age = data.get("age")  # Could be "25" or 25

    # This works in staging (maybe the test data is correct)
    # But fails in production when the LLM returns strings
    result = user_id * 2  # TypeError if user_id is "123"
    database_query = f"SELECT * FROM users WHERE age > {age}"  # SQL injection risk if age is string

    return {"processed_id": result, "query": database_query}

# Simulate what happens when an LLM returns inconsistent data
llm_response_1 = {"user_id": 123, "age": 25}  # ✅ Works (staging)
llm_response_2 = {"user_id": "123", "age": "25"}  # ❌ Fails silently (production)

print("Response 1 (staging):")
try:
    result = process_user_data_brittle(llm_response_1)
    print(f"  Success: {result}")
except Exception as e:
    print(f"  Error: {e}")

print("\nResponse 2 (production):")
try:
    result = process_user_data_brittle(llm_response_2)
    print(f"  Success: {result}")
except Exception as e:
    print(f"  Error: {type(e).__name__}: {e}")

Response 1 (staging):
  Success: {'processed_id': 246, 'query': 'SELECT * FROM users WHERE age > 25'}

Response 2 (production):
  Success: {'processed_id': '123123', 'query': 'SELECT * FROM users WHERE age > 25'}


## The Solution - Pydantic Validation

Pydantic catches these errors **immediately** at the boundary, before they spread through your system.

In [5]:
# Define the exact shape of your data
class UserData(BaseModel):
    """Technical spec: defines exactly what data we expect."""
    user_id: int = Field(..., description="User ID must be an integer")
    age: int = Field(..., ge=0, le=150, description="Age must be between 0 and 150")
    email: Optional[str] = Field(None, description="Optional email address")

    class Config:
        # Automatically convert strings to ints when possible
        # This handles provider differences gracefully
        str_strip_whitespace = True

def process_user_data_safe(data: dict):
    """With Pydantic, validation happens at the boundary."""
    try:
        # Validation happens here - fails fast if data is wrong
        user = UserData(**data)

        # Now we can trust the types
        result = user.user_id * 2
        database_query = f"SELECT * FROM users WHERE age > {user.age}"

        return {"processed_id": result, "query": database_query, "validated": user}
    except ValidationError as e:
        return {"error": "Validation failed", "details": str(e)}

# Same test cases
llm_response_1 = {"user_id": 123, "age": 25}
llm_response_2 = {"user_id": "123", "age": "25"}  # Strings - Pydantic converts them!
llm_response_3 = {"user_id": "abc", "age": "25"}  # Invalid - Pydantic catches it!

print("Response 1 (valid ints):")
result = process_user_data_safe(llm_response_1)
print(f"  {result}")

print("\nResponse 2 (string numbers - Pydantic converts):")
result = process_user_data_safe(llm_response_2)
print(f"  {result}")

print("\nResponse 3 (invalid data - Pydantic catches):")
result = process_user_data_safe(llm_response_3)
print(f"  {result}")

Response 1 (valid ints):
  {'processed_id': 246, 'query': 'SELECT * FROM users WHERE age > 25', 'validated': UserData(user_id=123, age=25, email=None)}

Response 2 (string numbers - Pydantic converts):
  {'processed_id': 246, 'query': 'SELECT * FROM users WHERE age > 25', 'validated': UserData(user_id=123, age=25, email=None)}

Response 3 (invalid data - Pydantic catches):
  {'error': 'Validation failed', 'details': "1 validation error for UserData\nuser_id\n  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='abc', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.12/v/int_parsing"}


/tmp/ipykernel_1461/2645436559.py:2: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class UserData(BaseModel):


## Provider Differences - The Real-World Problem

Different LLM providers return slightly different JSON structures. Without validation, you need custom parsing for each provider.

In [6]:
# Simulate different provider responses
class ProviderResponse(BaseModel):
    """Unified model that works across all providers."""
    task: str
    priority: int = Field(ge=1, le=5)
    tags: List[str] = Field(default_factory=list)
    metadata: dict = Field(default_factory=dict)

# Provider A: Returns nested structure
provider_a_response = {
    "task": "Write blog post",
    "priority": "3",  # String instead of int
    "tags": ["ai", "agents"],
    "metadata": {"source": "user"}
}

# Provider B: Returns flat structure with different field names
provider_b_response = {
    "task": "Write blog post",
    "priority": 3,
    "tags": "ai,agents",  # Comma-separated string instead of list
    "extra": {"source": "user"}  # Different key name
}

# Provider C: Returns valid JSON but missing fields
provider_c_response = {
    "task": "Write blog post",
    "priority": 3
    # Missing tags and metadata
}

def parse_provider_response_brittle(data: dict, provider: str):
    """Without Pydantic, you need custom parsing for each provider."""
    if provider == "A":
        priority = int(data.get("priority", 0))
        tags = data.get("tags", [])
    elif provider == "B":
        priority = data.get("priority", 0)
        tags = data.get("tags", "").split(",") if isinstance(data.get("tags"), str) else data.get("tags", [])
    else:
        priority = data.get("priority", 0)
        tags = data.get("tags", [])

    return {"task": data.get("task"), "priority": priority, "tags": tags}

def parse_provider_response_safe(data: dict):
    """With Pydantic, one model handles all providers."""
    # Pydantic automatically:
    # - Converts string "3" to int 3
    # - Handles missing fields (uses defaults)
    # - Validates types and constraints
    try:
        # Handle provider B's comma-separated tags
        if isinstance(data.get("tags"), str):
            data["tags"] = [tag.strip() for tag in data["tags"].split(",") if tag.strip()]

        # Handle provider B's different metadata key
        if "extra" in data and "metadata" not in data:
            data["metadata"] = data.pop("extra")

        return ProviderResponse(**data)
    except ValidationError as e:
        return {"error": str(e)}

print("Provider A (string priority):")
result = parse_provider_response_safe(provider_a_response)
print(f"  {result}")

print("\nProvider B (comma-separated tags, different key):")
result = parse_provider_response_safe(provider_b_response)
print(f"  {result}")

print("\nProvider C (missing fields):")
result = parse_provider_response_safe(provider_c_response)
print(f"  {result}")

Provider A (string priority):
  task='Write blog post' priority=3 tags=['ai', 'agents'] metadata={'source': 'user'}

Provider B (comma-separated tags, different key):
  task='Write blog post' priority=3 tags=['ai', 'agents'] metadata={'source': 'user'}

Provider C (missing fields):
  task='Write blog post' priority=3 tags=[] metadata={}


## Why Pydantic Matters for Agents

1. **Prevents Silent Failures**: Catches type mismatches at the boundary, not deep in your code
2. **Provider Consistency**: One model handles all provider variations (strings vs ints, different keys, etc.)
3. **Type Safety**: Your IDE and type checkers understand your data structure
4. **Self-Documenting**: The model is the spec - no separate documentation needed
5. **LangGraph Integration**: Validated state prevents corruption across agent steps

## Best Practices

- **Define models at boundaries**: Where data enters your system (LLM responses, API calls, user input)
- **Use Field constraints**: `ge`, `le`, `pattern` for validation rules
- **Handle provider differences**: Pydantic's type coercion handles common variations
- **Fail fast**: Let validation errors bubble up immediately, don't try to "fix" invalid data

# Integration of All Concepts

## Governance Config and Policy Selection

In [7]:
GOVERNANCE_CONFIG = """
# Runtime mode: "strict", "moderate", or "lenient"
mode: "strict"

# Mode-specific policies
policies:
  strict:
    max_retries: 3
    prompt:
      min_length: 50
      required_keywords: ["pydantic", "validation", "contract"]
      forbid_markdown: true  # Real constraint: no code blocks in prompts
      require_style_tag: true  # Must include style descriptor
      size_px_min: 512
      size_px_max: 1536
    image_output:
      allowed_mime: ["image/png"]
      max_bytes: 2000000
    db:
      serialize_mode: "json"
      schema_version: 1

  moderate:
    max_retries: 2
    prompt:
      min_length: 30
      required_keywords: ["pydantic"]
      forbid_markdown: false
      require_style_tag: false
      size_px_min: 256
      size_px_max: 2048
    image_output:
      allowed_mime: ["image/png", "image/jpeg"]
      max_bytes: 5000000
    db:
      serialize_mode: "json"
      schema_version: 1

  lenient:
    max_retries: 1
    prompt:
      min_length: 20
      required_keywords: []
      forbid_markdown: false
      require_style_tag: false
      size_px_min: 256
      size_px_max: 4096
    image_output:
      allowed_mime: ["image/png", "image/jpeg", "image/webp"]
      max_bytes: 10000000
    db:
      serialize_mode: "json"
      schema_version: 1
"""

cfg: DictConfig = OmegaConf.create(GOVERNANCE_CONFIG)

def get_active_policy(cfg: DictConfig) -> DictConfig:
    """Get the active policy based on current mode."""
    mode = str(cfg.mode)
    return cfg.policies[mode]


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def hash_policy(policy_dict: dict[str, Any]) -> str:
    """Generate deterministic hash of policy for versioning."""
    canonical = json.dumps(policy_dict, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return sha256_text(canonical)[:16]


## Contracts and Gates (with Caching)

In [8]:
# Cache Pydantic models by (mode, policy_hash) to avoid stale validators
# Returns: (ImagePrompt, GeneratedImageToolResult, GeneratedImageRecord)
_model_cache: dict[tuple[str, str], tuple[type[BaseModel], type[BaseModel], type[BaseModel]]] = {}


def build_image_prompt_model(policy_dict: dict[str, Any], mode: str, policy_hash: str):
    """Factory: Build Pydantic model from policy dict. Cached by mode and policy_hash."""
    prompt_cfg = policy_dict["prompt"]
    min_len = int(prompt_cfg["min_length"])
    required = [str(x).lower() for x in prompt_cfg.get("required_keywords", [])]
    size_min = int(prompt_cfg["size_px_min"])
    size_max = int(prompt_cfg["size_px_max"])
    forbid_md = bool(prompt_cfg.get("forbid_markdown", False))
    require_style = bool(prompt_cfg.get("require_style_tag", False))

    class ImagePrompt(BaseModel):
        model_config = {"title": f"ImagePrompt_{mode}_{policy_hash}"}  # Stable schema refs

        prompt: str = Field(..., min_length=min_len)
        size_px: int = Field(default=1024, ge=size_min, le=size_max)

        @field_validator("prompt")
        @classmethod
        def enforce_keywords(cls, v: str) -> str:
            if required:
                low = v.lower()
                missing = [k for k in required if k not in low]
                if missing:
                    raise ValueError(f"Prompt missing required keywords: {missing}")
            return v

        @field_validator("prompt")
        @classmethod
        def forbid_markdown_blocks(cls, v: str) -> str:
            # Only check for code blocks (triple backticks), not inline formatting
            if forbid_md and "```" in v:
                raise ValueError("Prompt must not contain markdown code blocks")
            return v

        @field_validator("prompt")
        @classmethod
        def require_style_descriptor(cls, v: str) -> str:
            if require_style:
                style_tags = ["minimal", "vector", "illustration", "photo", "diagram"]
                low = v.lower()
                if not any(tag in low for tag in style_tags):
                    raise ValueError(f"Prompt must include a style tag: {style_tags}")
            return v

    return ImagePrompt


def build_generated_image_model(policy_dict: dict[str, Any], mode: str, policy_hash: str):
    """Factory: Build Pydantic model from policy dict. Cached by mode and policy_hash."""
    image_cfg = policy_dict["image_output"]
    allowed_mime = {str(x) for x in image_cfg["allowed_mime"]}
    max_bytes = int(image_cfg["max_bytes"])

    class GeneratedImageToolResult(BaseModel):
        """Tool result contract: validates output from image generation tool."""
        model_config = {"title": f"GeneratedImageToolResult_{mode}_{policy_hash}"}  # Stable schema refs

        asset_id: str = Field(..., min_length=8)
        mime_type: str
        bytes_size: int = Field(..., ge=1, le=max_bytes)
        width_px: int = Field(..., ge=1)
        height_px: int = Field(..., ge=1)

        @field_validator("mime_type")
        @classmethod
        def validate_mime(cls, v: str) -> str:
            if v not in allowed_mime:
                raise ValueError(f"mime_type must be one of {sorted(allowed_mime)}")
            return v

    return GeneratedImageToolResult


def build_generated_image_record_model(policy_dict: dict[str, Any], mode: str, policy_hash: str):
    """Factory: Build DB record model for image storage. Separate from tool result contract."""
    image_cfg = policy_dict["image_output"]
    allowed_mime = {str(x) for x in image_cfg["allowed_mime"]}
    max_bytes = int(image_cfg["max_bytes"])

    class GeneratedImageRecord(BaseModel):
        """DB record contract: validates stored image fields. May drift independently from tool contract."""
        model_config = {"title": f"GeneratedImageRecord_{mode}_{policy_hash}"}  # Stable schema refs

        asset_id: str = Field(..., min_length=8)
        mime_type: str
        bytes_size: int = Field(..., ge=1, le=max_bytes)
        width_px: int = Field(..., ge=1)
        height_px: int = Field(..., ge=1)

        @field_validator("mime_type")
        @classmethod
        def validate_mime(cls, v: str) -> str:
            if v not in allowed_mime:
                raise ValueError(f"mime_type must be one of {sorted(allowed_mime)}")
            return v

    return GeneratedImageRecord


def get_models_for_policy(policy_dict: dict[str, Any], mode: str, policy_hash: str):
    """Get cached models for a policy. Cache key includes policy_hash to avoid stale validators.

    Returns: (ImagePrompt, GeneratedImageToolResult, GeneratedImageRecord)
    """
    cache_key = (mode, policy_hash)
    if cache_key not in _model_cache:
        _model_cache[cache_key] = (
            build_image_prompt_model(policy_dict, mode, policy_hash),
            build_generated_image_model(policy_dict, mode, policy_hash),  # Tool result contract
            build_generated_image_record_model(policy_dict, mode, policy_hash),  # DB record contract
        )
    return _model_cache[cache_key]


class MediaRecord(BaseModel):
    """DB record contract with schema versioning and audit trail.

    Note: image field accepts any BaseModel instance (GeneratedImageRecord is built dynamically
    per policy mode, but we enforce it's a validated Pydantic model via type annotation).
    """
    record_id: str
    created_at: str
    prompt_hash: str
    prompt_text: str
    image: BaseModel  # Typed contract (GeneratedImageRecord instance), validated before construction
    record_schema_version: int = Field(default=1)
    policy_hash: str  # Hash of policy snapshot for audit
    schema_hash: str  # Combined hash of schemas + policy semantics (for migration safety)
    run_id: str  # Run identifier tying DB record to LangGraph thread_id and checkpointer state

    def to_json_dict(self) -> dict[str, Any]:
        """Convert to JSON-compatible dict for DB storage using single point of truth."""
        # Defensive check: ensure image is a BaseModel (Pydantic validates, but explicit is clearer)
        if not isinstance(self.image, BaseModel):
            raise TypeError(f"image must be a BaseModel instance, got {type(self.image)}")

        # Use model_dump for entire record, but explicitly serialize nested image contract
        # This makes the contract boundary clear: we intentionally serialize the validated image
        data = self.model_dump(mode="json")
        data["image"] = self.image.model_dump(mode="json")  # Explicit contract serialization
        return data


class GraphState(TypedDict):
    messages: Annotated[list[BaseMessage], add]
    execution_log: Annotated[list[str], add]  # Uses additive reducer: every node appends

    image_prompt: dict[str, Any]
    generated_image: dict[str, Any]

    retry_count: int
    last_error: str
    prompt_ok: bool  # Explicit validation status
    image_ok: bool   # Explicit validation status

    db_record: dict[str, Any]
    active_policy: dict[str, Any]  # Pinned policy for this run
    active_mode: str  # Mode snapshot for this run
    policy_hash: str  # Hash of pinned policy (for cache key and audit)
    schema_hash: str  # Combined hash of schemas + policy semantics (for migration safety)
    max_retries: int  # Pinned max_retries from policy (avoid repeated lookups)
    run_id: str  # Run identifier tying DB record to LangGraph thread_id and checkpointer state


## Build the Nodes

In [9]:
def node_initialize_policy(state: GraphState):
    """Pin policy at START to ensure deterministic validation throughout run."""
    policy = get_active_policy(cfg)
    policy_dict = OmegaConf.to_container(policy, resolve=True)
    mode = str(cfg.mode)

    # Compute policy hash for caching and audit
    policy_hash = hash_policy(policy_dict)

    # Pre-build and cache models for this policy
    ImagePrompt, GeneratedImageToolResult, GeneratedImageRecord = get_models_for_policy(policy_dict, mode, policy_hash)

    # Extract policy semantics that affect validation but aren't in JSON Schema
    # (e.g., required_keywords, forbid_markdown, require_style_tag)
    # Normalize aggressively for hash stability
    prompt_cfg = policy_dict["prompt"]
    policy_semantics = {
        "required_keywords": sorted([str(x).lower() for x in prompt_cfg.get("required_keywords", [])]),  # Normalized
        "forbid_markdown": bool(prompt_cfg.get("forbid_markdown", False)),
        "require_style_tag": bool(prompt_cfg.get("require_style_tag", False)),
        "max_retries": int(policy_dict["max_retries"]),
        # If style_tags list exists in config, normalize it here too
        # "style_tags": sorted([str(x).lower() for x in prompt_cfg.get("style_tags", [])]),
    }

    # Compute combined schema hash: schemas + policy semantics for migration safety
    # This ensures schema_hash changes when validation behavior changes, even if JSON Schema doesn't
    prompt_schema = ImagePrompt.model_json_schema()
    tool_schema = GeneratedImageToolResult.model_json_schema()
    record_schema = GeneratedImageRecord.model_json_schema()
    combined = {
        "schemas": {
            "ImagePrompt": prompt_schema,
            "GeneratedImageToolResult": tool_schema,
            "GeneratedImageRecord": record_schema,
        },
        "policy_semantics": policy_semantics,
    }
    schema_hash = sha256_text(
        json.dumps(combined, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    )[:16]

    # Use run_id from state if present (set from thread_id in config), otherwise generate
    run_id = state.get("run_id") or f"run_{sha256_text(f'{mode}_{datetime.now(timezone.utc).isoformat()}')[:12]}"

    return {
        "active_policy": policy_dict,
        "active_mode": mode,
        "policy_hash": policy_hash,
        "schema_hash": schema_hash,
        "max_retries": int(policy_dict["max_retries"]),  # Store for routing (avoid repeated lookups)
        "run_id": run_id,  # Run identifier for DB record and checkpoint tracing
        "prompt_ok": False,
        "image_ok": False,
        "execution_log": [f"Initialized: mode={mode}, policy_hash={policy_hash}, schema_hash={schema_hash}, run_id={run_id}"],
    }


def node_content_writer(state: GraphState):
    """Agent A: Writes content draft."""
    return {
        "execution_log": ["A: Content draft written"],
    }


def node_propose_image_prompt(state: GraphState):
    """Agent B: Proposes image prompt (may be invalid initially)."""
    return {
        "image_prompt": {
            "prompt": "Short",  # Will fail validation
            "size_px": 1024,
        },
        "execution_log": ["B: Proposed image prompt"],
    }


def node_prompt_validation_gate(state: GraphState):
    """Validation Gate: Enforces Pydantic contract with retry logic."""
    policy_dict = state["active_policy"]
    mode = state["active_mode"]
    policy_hash = state["policy_hash"]
    ImagePrompt, _, _ = get_models_for_policy(policy_dict, mode, policy_hash)

    raw_prompt = state.get("image_prompt", {})
    retries = int(state.get("retry_count", 0))
    max_retries = int(state["max_retries"])  # Use pinned value from state

    try:
        valid_prompt = ImagePrompt(**raw_prompt)
        return {
            "image_prompt": valid_prompt.model_dump(),
            "retry_count": 0,
            "last_error": "",
            "prompt_ok": True,  # Explicit success flag
            "execution_log": [f"C: Prompt validated on attempt {retries + 1}"],
        }
    except ValidationError as e:
        error_msg = e.errors()[0]["msg"]

        # Increment retry count first, then check (makes log numbering clearer)
        new_retries = retries + 1

        if new_retries > max_retries:
            return {
                "retry_count": new_retries,  # Update so routing can detect exceeded state
                "prompt_ok": False,
                "execution_log": [f"C: Max retries ({max_retries}) exceeded on attempt {new_retries}. Last error: {error_msg}"],
                "last_error": error_msg,
            }

        return {
            "retry_count": new_retries,
            "last_error": error_msg,
            "prompt_ok": False,
            "execution_log": [f"C: Validation failed on attempt {new_retries}: {error_msg}"],
        }


def node_refine_prompt(state: GraphState):
    """Agent D: Refines prompt based on validation error."""
    policy_dict = state["active_policy"]
    last_error = state.get("last_error", "")

    prompt_cfg = policy_dict["prompt"]
    min_len = int(prompt_cfg["min_length"])
    required = [str(x) for x in prompt_cfg.get("required_keywords", [])]
    require_style = bool(prompt_cfg.get("require_style_tag", False))

    # Build prompt that satisfies all requirements
    style_part = "minimal vector illustration" if require_style else ""
    refined_prompt = {
        "prompt": f"A high-resolution technical illustration showing Pydantic validation contracts in a multi-agent system. {style_part} " +
                  " ".join([f"Focus on {kw}." for kw in required]) +
                  f" Clean style. Minimum {min_len} characters.",
        "size_px": 1024,
    }

    return {
        "image_prompt": refined_prompt,
        "execution_log": ["D: Prompt refined based on validation error"],
    }


def node_generate_image(state: GraphState):
    """Agent E: Calls image generation tool."""
    prompt = state["image_prompt"]["prompt"]

    tool_output = {
        "asset_id": sha256_text(prompt)[:12],
        "mime_type": "image/png",
        "bytes_size": 150000,
        "width_px": state["image_prompt"]["size_px"],
        "height_px": state["image_prompt"]["size_px"],
    }

    return {
        "generated_image": tool_output,
        "execution_log": ["E: Image generation tool executed"],
    }


def node_image_output_validation_gate(state: GraphState):
    """Validation Gate: Validates tool output before DB storage."""
    policy_dict = state["active_policy"]
    mode = state["active_mode"]
    policy_hash = state["policy_hash"]
    _, GeneratedImageToolResult, _ = get_models_for_policy(policy_dict, mode, policy_hash)

    raw_image = state.get("generated_image", {})

    try:
        valid_image = GeneratedImageToolResult(**raw_image)
        return {
            "generated_image": valid_image.model_dump(),
            "image_ok": True,  # Explicit success flag
            "execution_log": ["F: Image tool output validated"],
        }
    except ValidationError as e:
        error_msg = e.errors()[0]["msg"]
        return {
            "generated_image": {},
            "image_ok": False,
            "execution_log": [f"F: Image tool output rejected: {error_msg}"],
        }


def node_prepare_db_record(state: GraphState):
    """Prepares validated record for DB storage with JSON serialization."""
    policy_dict = state["active_policy"]
    mode = state["active_mode"]
    policy_hash = state["policy_hash"]
    schema_hash = state["schema_hash"]
    _, _, GeneratedImageRecord = get_models_for_policy(policy_dict, mode, policy_hash)

    prompt_text = state["image_prompt"]["prompt"]
    prompt_hash = sha256_text(prompt_text)

    # Convert tool result to DB record contract (may have different validation rules)
    # In this example they're identical, but in production they can drift independently
    image_record = GeneratedImageRecord(**state["generated_image"])

    record = MediaRecord(
        record_id=f"media_{prompt_hash[:12]}",
        created_at=datetime.now(timezone.utc).isoformat(),
        prompt_hash=prompt_hash,
        prompt_text=prompt_text,
        image=image_record,  # Typed contract (GeneratedImageRecord BaseModel), not dict
        record_schema_version=int(policy_dict["db"]["schema_version"]),
        policy_hash=policy_hash,  # Use stored hash from init
        schema_hash=schema_hash,  # Use stored hash from init
        run_id=state["run_id"],  # Run identifier tying DB record to thread_id and checkpointer
    )

    db_dict = record.to_json_dict()

    return {
        "db_record": db_dict,
        "execution_log": ["G: DB record prepared with JSON serialization"],
    }


def node_persist_to_db(state: GraphState):
    """Persists validated, JSON-serialized record to DB."""
    record = state["db_record"]
    return {
        "execution_log": [f"H: Persisted to DB: record_id={record['record_id']}"],
    }


def should_retry_prompt(state: GraphState) -> str:
    """Route based on explicit validation status."""
    if state.get("prompt_ok", False):
        return "ok"

    max_retries = int(state["max_retries"])  # Use pinned value from state
    retries = int(state.get("retry_count", 0))

    # Check if we've exceeded max retries (node increments first, then checks)
    if retries > max_retries:
        return "end"
    return "fix"


def should_continue_with_image(state: GraphState) -> str:
    """Route based on explicit validation status."""
    return "ok" if state.get("image_ok", False) else "end"


## Build the Graph

In [10]:
memory = InMemorySaver()
workflow = StateGraph(GraphState)

workflow.add_node("init_policy", node_initialize_policy)
workflow.add_node("writer", node_content_writer)
workflow.add_node("propose_prompt", node_propose_image_prompt)
workflow.add_node("prompt_gate", node_prompt_validation_gate)
workflow.add_node("refine_prompt", node_refine_prompt)
workflow.add_node("generate_image", node_generate_image)
workflow.add_node("image_gate", node_image_output_validation_gate)
workflow.add_node("prepare_db", node_prepare_db_record)
workflow.add_node("persist_db", node_persist_to_db)

workflow.add_edge(START, "init_policy")
workflow.add_edge("init_policy", "writer")
workflow.add_edge("writer", "propose_prompt")
workflow.add_edge("propose_prompt", "prompt_gate")

workflow.add_conditional_edges(
    "prompt_gate",
    should_retry_prompt,
    {"fix": "refine_prompt", "ok": "generate_image", "end": END},
)
workflow.add_edge("refine_prompt", "prompt_gate")

workflow.add_edge("generate_image", "image_gate")
workflow.add_conditional_edges(
    "image_gate",
    should_continue_with_image,
    {"ok": "prepare_db", "end": END},
)

workflow.add_edge("prepare_db", "persist_db")
workflow.add_edge("persist_db", END)

app = workflow.compile(checkpointer=memory)


## Run the App

In [11]:
print("=" * 80)
print("PRODUCTION SYSTEM DEMONSTRATION")
print("=" * 80)
# Note: We print from cfg here for initial display, but actual execution uses pinned values from state
policy = get_active_policy(cfg)
print(f"Initial Mode: {cfg.mode}")
print(f"Max Retries: {policy.max_retries}")
print(f"Prompt Min Length: {policy.prompt.min_length}")
print(f"Required Keywords: {policy.prompt.required_keywords}")
print(f"Forbid Markdown: {policy.prompt.forbid_markdown}")
print(f"Require Style Tag: {policy.prompt.require_style_tag}")
print("=" * 80)
print()

config = {"configurable": {"thread_id": "production_run_001"}}

# Set run_id from thread_id to tie DB record to LangGraph checkpoint state
initial_state: GraphState = {
    "messages": [HumanMessage(content="Generate a blog cover image")],
    "execution_log": [],
    "image_prompt": {},
    "generated_image": {},
    "retry_count": 0,
    "last_error": "",
    "prompt_ok": False,
    "image_ok": False,
    "db_record": {},
    "active_policy": {},
    "active_mode": "",
    "policy_hash": "",
    "schema_hash": "",
    "max_retries": 0,
    "run_id": config["configurable"]["thread_id"],  # Tie DB record to LangGraph thread_id
}

print("--- Running Complete Workflow ---")
result = app.invoke(initial_state, config)

print("\n--- Execution Log ---")
for line in result["execution_log"]:
    print(f"  {line}")

print("\n--- DB Record (JSON-Compatible with Versioning) ---")
print(json.dumps(result["db_record"], indent=2, ensure_ascii=False))

print("\n--- Policy Snapshot (Pinned at Start) ---")
print(f"  Mode: {result['active_mode']}")  # Use pinned value from state, not cfg.mode
print(f"  Max Retries: {result.get('max_retries', 'N/A')}")  # Use pinned value from state
print(f"  Run ID: {result.get('run_id', 'N/A')}")  # Run identifier for tracing
print(f"  Policy Hash: {result['db_record'].get('policy_hash', 'N/A')}")
print(f"  Schema Hash: {result['db_record'].get('schema_hash', 'N/A')}")
print(f"  Schema Version: {result['db_record'].get('record_schema_version', 'N/A')}")

print("\n--- Fault Tolerance: Checkpoint Inspection ---")
snapshot = app.get_state(config)
print(f"  Last node: {snapshot.next if hasattr(snapshot, 'next') else 'N/A'}")
print(f"  Policy pinned: {bool(snapshot.values.get('active_policy'))}")
print(f"  Run ID: {snapshot.values.get('run_id', 'N/A')}")  # Ties DB record to checkpoint
print(f"  Policy hash: {snapshot.values.get('policy_hash', 'N/A')}")
print(f"  Schema hash: {snapshot.values.get('schema_hash', 'N/A')}")

PRODUCTION SYSTEM DEMONSTRATION
Initial Mode: strict
Max Retries: 3
Prompt Min Length: 50
Required Keywords: ['pydantic', 'validation', 'contract']
Forbid Markdown: True
Require Style Tag: True

--- Running Complete Workflow ---

--- Execution Log ---
  Initialized: mode=strict, policy_hash=64a35d3ba43027ae, schema_hash=7720b7473856aab3, run_id=production_run_001
  A: Content draft written
  B: Proposed image prompt
  C: Validation failed on attempt 1: String should have at least 50 characters
  D: Prompt refined based on validation error
  C: Prompt validated on attempt 2
  E: Image generation tool executed
  F: Image tool output validated
  G: DB record prepared with JSON serialization
  H: Persisted to DB: record_id=media_ad4f8f9896c2

--- DB Record (JSON-Compatible with Versioning) ---
{
  "record_id": "media_ad4f8f9896c2",
  "created_at": "2026-06-10T12:39:15.804294+00:00",
  "prompt_hash": "ad4f8f9896c2184e5f2a7861d1eac320a0927a12753e12d489397a324a8d90ae",
  "prompt_text": "A hig

## Pydantic vs Instructor: Boundary Protection vs. Recovery at the Source

In your LangGraph workflow, Pydantic acts as a boundary gate. It prevents invalid data from propagating, but recovery logic still lives in the orchestration layer, for example a separate refine node and explicit retry routing.

Instructor is useful when you want to move that recovery closer to the source. Instead of treating structured output validation as a separate workflow concern, Instructor couples schema enforcement with generation. The model is repeatedly asked to produce an output that satisfies the same pinned Pydantic contract, using the validation error as feedback. This reduces graph branching, removes the need for specific "fixer" nodes, and makes structured output behavior more consistent across different models and providers.

In other words, Pydantic protects the system boundary. Instructor reduces the work required to consistently hit that boundary.

# Instructor: Recovery at the Source

### Same Pydantic model as before (reuse the policy-driven model)

In [13]:
policy = get_active_policy(cfg)
policy_dict = OmegaConf.to_container(policy, resolve=True)
mode = str(cfg.mode)
policy_hash = hash_policy(policy_dict)

ImagePrompt, _, _ = get_models_for_policy(policy_dict, mode, policy_hash)

# Create Instructor client using OpenAI client (compatible with OpenRouter)
# Instructor works by patching the provider's create() method
# Architecture: Pydantic Model → Provider Handler → Dispatcher → Provider Client → Retry Layer → Instructor (patched)
from openai import OpenAI
model_slug = os.environ.get("OPENROUTER_MODEL", "anthropic/claude-3.5-haiku")

# Create OpenAI client configured for OpenRouter
client = instructor.from_provider(
    f"openrouter/{model_slug}",
    base_url="https://openrouter.ai/api/v1",
    mode=instructor.Mode.TOOLS,
)

print("=" * 80)
print("Automatic Recovery at Generation Time")
print("=" * 80)
print(f"Policy Mode: {mode}")
print(f"Max Retries: {policy.max_retries}")
print(f"Prompt Min Length: {policy.prompt.min_length}")
print(f"Required Keywords: {policy.prompt.required_keywords}")
print("=" * 80)
print()

# Instructor automatically retries with validation error feedback
# No need for separate refine nodes or conditional routing
print("--- Generating Image Prompt with Instructor ---")
print("Instructor will automatically retry if validation fails, using error messages as feedback.")
print()

try:
    # Instructor handles validation internally - no manual retry logic needed
    # The patched client's chat.completions.create() now supports response_model and max_retries
    # Flow: create() → retry layer → process_response → validation → reask if needed
    # When validation fails, Instructor's handle_reask_kwargs() appends error feedback
    result = client.create(                       # was client.chat.completions.create(...)
        messages=[
            {"role": "user",
            "content": "Generate an image prompt for a blog cover about Pydantic validation in multi-agent systems."}
        ],
        response_model=ImagePrompt,               # triggers schema/tool wiring + parsing
        max_retries=policy.max_retries,           # tenacity-backed validation retries
        temperature=0.2,
    )

    # Instructor returns the parsed Pydantic model directly (with _raw_response attached)
    print(f"✓ Successfully generated valid prompt after automatic retries:")
    print(f"  Prompt: {result.prompt[:100]}...")
    print(f"  Length: {len(result.prompt)} characters")
    print(f"  Size: {result.size_px}px")
    # Access raw provider response if needed: result._raw_response
    print()
    print("Key difference: No separate 'refine_prompt' node needed!")
    print("Instructor handled validation and retry logic internally.")

except Exception as e:
    print(f"✗ Failed after max retries: {e}")



Automatic Recovery at Generation Time
Policy Mode: strict
Max Retries: 3
Prompt Min Length: 50
Required Keywords: ['pydantic', 'validation', 'contract']

--- Generating Image Prompt with Instructor ---
Instructor will automatically retry if validation fails, using error messages as feedback.

✓ Successfully generated valid prompt after automatic retries:
  Prompt: A futuristic digital landscape showcasing a contract-based validation system for multi-agent AI inte...
  Length: 568 characters
  Size: 1024px

Key difference: No separate 'refine_prompt' node needed!
Instructor handled validation and retry logic internally.


# Simplified LangGraph Workflow with Instructor

In [16]:
# Simplified state - no retry_count, last_error, or prompt_ok needed
class SimplifiedGraphState(TypedDict):
    messages: Annotated[list[BaseMessage], add]
    execution_log: Annotated[list[str], add]
    image_prompt: dict[str, Any]
    generated_image: dict[str, Any]

def node_propose_prompt_with_instructor(state: SimplifiedGraphState):
    """Generate prompt using Instructor - validation and retry happen here."""
    # Instructor handles validation internally, so we get a valid prompt directly
    # The patched client automatically retries with validation error feedback
    result = client.chat.completions.create(
        model=os.environ.get("OPENROUTER_MODEL", "anthropic/claude-3.5-haiku"),
        messages=[
            {
                "role": "user",
                "content": "Generate an image prompt for a blog cover about Pydantic validation in multi-agent systems."
            }
        ],
        response_model=ImagePrompt,
        max_retries=policy.max_retries,
        temperature=0.2,
    )

    return {
        "image_prompt": result.model_dump(),
        "execution_log": ["B: Generated valid prompt with Instructor (automatic retry on validation errors)"],
    }

def node_generate_image_simple(state: SimplifiedGraphState):
    """Generate image - no validation gate needed if Instructor ensures valid input."""
    prompt = state["image_prompt"]["prompt"]

    tool_output = {
        "asset_id": sha256_text(prompt)[:12],
        "mime_type": "image/png",
        "bytes_size": 150000,
        "width_px": state["image_prompt"]["size_px"],
        "height_px": state["image_prompt"]["size_px"],
    }

    return {
        "generated_image": tool_output,
        "execution_log": ["E: Image generation tool executed"],
    }

# Build simplified graph - no refine node, no conditional retry routing
simplified_workflow = StateGraph(SimplifiedGraphState)

simplified_workflow.add_node("propose_prompt", node_propose_prompt_with_instructor)
simplified_workflow.add_node("generate_image", node_generate_image_simple)

# Simple linear flow - no branching for retries
simplified_workflow.add_edge(START, "propose_prompt")
simplified_workflow.add_edge("propose_prompt", "generate_image")
simplified_workflow.add_edge("generate_image", END)

simplified_app = simplified_workflow.compile(checkpointer=MemorySaver())

print("=" * 80)
print("Running Simplified LangGraph Workflow with Instructor")
print("=" * 80)
print()

initial_state = {
    "messages": [HumanMessage(content="Generate a blog cover image")],
    "execution_log": [],
    "image_prompt": {},
    "generated_image": {},
}

result = simplified_app.invoke(initial_state, {"configurable": {"thread_id": "instructor_demo"}})

print("--- Execution Log ---")
for line in result["execution_log"]:
    print(f"  {line}")

print()
print("--- Generated Prompt ---")
print(f"  Prompt: {result['image_prompt'].get('prompt', 'N/A')[:100]}...")
print(f"  Length: {len(result['image_prompt'].get('prompt', ''))} characters")

print()
print("=" * 80)
print("GRAPH COMPARISON:")
print("=" * 80)
print("Previous (LangGraph + Pydantic):")
print("  START → init_policy → writer → propose_prompt → prompt_gate")
print("  prompt_gate → [fix: refine_prompt → prompt_gate] | [ok: generate_image]")
print("  generate_image → image_gate → [ok: prepare_db] | [end: END]")
print("  prepare_db → persist_db → END")
print()
print("With Instructor:")
print("  START → propose_prompt → generate_image → END")
print()
print("Nodes removed: init_policy, writer, prompt_gate, refine_prompt,")
print("               image_gate, prepare_db, persist_db")
print("Conditional edges removed: retry routing logic")
print("=" * 80)

Running Simplified LangGraph Workflow with Instructor

--- Execution Log ---
  B: Generated valid prompt with Instructor (automatic retry on validation errors)
  E: Image generation tool executed

--- Generated Prompt ---
  Prompt: A futuristic digital landscape [vector] showcasing a data contract validation process in a multi-age...
  Length: 456 characters

GRAPH COMPARISON:
Previous (LangGraph + Pydantic):
  START → init_policy → writer → propose_prompt → prompt_gate
  prompt_gate → [fix: refine_prompt → prompt_gate] | [ok: generate_image]
  generate_image → image_gate → [ok: prepare_db] | [end: END]
  prepare_db → persist_db → END

With Instructor:
  START → propose_prompt → generate_image → END

Nodes removed: init_policy, writer, prompt_gate, refine_prompt,
               image_gate, prepare_db, persist_db
Conditional edges removed: retry routing logic
